# KisanAI - Fertilizer Agent

KisanAI / Sampoorn Kisan AI Sahayak

## Goal
Train a real multi-output ML model on Soil Health Card data to classify nutrient status
(N, P, K, S, Zn, B, Fe, Cu, Mn -> Low/Medium/High) from soil context + lab values,
then convert those predictions into an actual fertilizer dosage recommendation.

Why ML instead of a fixed lookup table: the government's Soil Health Card ratings are
threshold-based per nutrient, but real deficiency also depends on interacting factors
(soil type, pH, organic carbon, EC) that a single-variable threshold ignores. A model
learns these interactions and generalizes better to soil types/regions not in a fixed table.

The final dosage numbers stay rule-based (standard agronomic tables) on top of the ML
classification -- this hybrid pattern matches the rest of the KisanAI architecture.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib

## Upload the Soil Health Card dataset
Upload `soil_health_card_sample.csv` (or your full/real dataset with the same columns).

In [ ]:
from google.colab import files
uploaded = files.upload()

import glob, pandas as pd
csvs = glob.glob("/content/**/*.csv", recursive=True)
print("CSV files found:", csvs)

CSV_PATH = csvs[0]
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nSoil types:", df['Soil_Type'].unique())
print("Seasons:", df['Season'].unique())

## Define features and targets

Features: soil context (Soil_Type, Season, pH, EC, OC) + raw lab values (kg/ha or ppm per nutrient).
Targets: the 9 nutrient Rating columns (multi-output classification -- one model predicts all 9).

In [ ]:
NUTRIENTS = ["N", "P", "K", "S", "Zn", "B", "Fe", "Cu", "Mn"]

# Raw value column for each nutrient (kg/ha for N/P/K, ppm for the rest)
VALUE_COLS = []
for nut in NUTRIENTS:
    candidates = [c for c in df.columns if c.startswith(nut + "_") and "Rating" not in c]
    VALUE_COLS.append(candidates[0])

TARGET_COLS = [f"{nut}_Rating" for nut in NUTRIENTS]

CONTEXT_NUMERIC = ["pH", "EC_dS_per_m", "OC_percent"]
CONTEXT_CATEGORICAL = ["Soil_Type", "Season"]

print("Value columns:", VALUE_COLS)
print("Target columns:", TARGET_COLS)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import numpy as np

# One-hot encode Soil_Type + Season
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
cat_encoded = encoder.fit_transform(df[CONTEXT_CATEGORICAL])
cat_encoded_df = pd.DataFrame(
    cat_encoded, columns=encoder.get_feature_names_out(CONTEXT_CATEGORICAL)
)

X = pd.concat(
    [df[CONTEXT_NUMERIC].reset_index(drop=True),
     df[VALUE_COLS].reset_index(drop=True),
     cat_encoded_df],
    axis=1
)

# Label-encode each target column separately (needed for multi-output sklearn models)
target_encoders = {}
y_encoded = pd.DataFrame()
for col in TARGET_COLS:
    le = LabelEncoder()
    y_encoded[col] = le.fit_transform(df[col])
    target_encoders[col] = le

print("X shape:", X.shape)
print("y shape:", y_encoded.shape)
X.head()

## Train/test split and model training

Using `MultiOutputClassifier` wrapping `RandomForestClassifier` -- trains one Random Forest
per nutrient internally, but you call it as a single model.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

base_model = RandomForestClassifier(n_estimators=300, random_state=42)
model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)

print("Training complete.")

In [ ]:
# Evaluate each nutrient separately -- a single overall accuracy number would hide
# whether the model is weak on a specific nutrient (e.g. rare micronutrient classes)
y_pred = model.predict(X_test)
y_pred_df = pd.DataFrame(y_pred, columns=TARGET_COLS)

for i, col in enumerate(TARGET_COLS):
    acc = accuracy_score(y_test[col], y_pred_df[col])
    print(f"{col}: accuracy = {acc:.3f}")

print("\nDetailed report for N_Rating (repeat for others as needed):")
le = target_encoders["N_Rating"]
print(classification_report(
    y_test["N_Rating"], y_pred_df["N_Rating"],
    labels=range(len(le.classes_)), target_names=le.classes_, zero_division=0
))

## Fertilizer dosage rule-engine

Standard agronomic dosage adjustments per rating level. These numbers are illustrative --
replace with your state agriculture department's actual recommendation table before using
this for real advisory output.

In [ ]:
# kg/ha adjustment guidance per nutrient and rating level (illustrative starting values)
DOSAGE_TABLE = {
    "N": {"Low": "Apply full recommended Urea dose (~120 kg/ha) + split application",
          "Medium": "Apply 75% of recommended Urea dose",
          "High": "Reduce Urea to 50% of recommended dose, avoid over-application"},
    "P": {"Low": "Apply full DAP dose (~60 kg/ha)",
          "Medium": "Apply 60% of recommended DAP dose",
          "High": "Skip additional phosphorus this season"},
    "K": {"Low": "Apply full MOP dose (~40 kg/ha)",
          "Medium": "Apply 50% of recommended MOP dose",
          "High": "Skip additional potassium this season"},
    "S": {"Low": "Apply Gypsum or Sulphur-coated fertilizer (~20 kg/ha S)",
          "Medium": "Apply reduced Sulphur dose",
          "High": "No sulphur application needed"},
    "Zn": {"Low": "Apply Zinc Sulphate (~25 kg/ha)",
           "Medium": "Apply maintenance Zinc dose (~10 kg/ha)",
           "High": "No zinc application needed"},
    "B": {"Low": "Apply Borax (~2 kg/ha, avoid contact with seed)",
          "Medium": "Apply maintenance Boron dose",
          "High": "No boron application needed -- risk of toxicity"},
    "Fe": {"Low": "Foliar spray of Ferrous Sulphate (0.5%)",
           "Medium": "Monitor; foliar spray optional",
           "High": "No iron application needed"},
    "Cu": {"Low": "Apply Copper Sulphate (~5 kg/ha)",
           "Medium": "Apply maintenance Copper dose",
           "High": "No copper application needed"},
    "Mn": {"Low": "Foliar spray of Manganese Sulphate (0.5%)",
           "Medium": "Monitor; foliar spray optional",
           "High": "No manganese application needed"},
}

def dosage_recommendation(ratings: dict) -> dict:
    """ratings: {'N': 'Low', 'P': 'Medium', ...} -> {'N': 'advice text', ...}"""
    return {
        nut: DOSAGE_TABLE.get(nut, {}).get(rating, "No guidance available")
        for nut, rating in ratings.items()
    }

## Save model + encoders

In [ ]:
import joblib, os

os.makedirs("/content/kisanai_models", exist_ok=True)

artifact = {
    "model": model,
    "encoder": encoder,
    "target_encoders": target_encoders,
    "context_numeric": CONTEXT_NUMERIC,
    "context_categorical": CONTEXT_CATEGORICAL,
    "value_cols": VALUE_COLS,
    "target_cols": TARGET_COLS,
    "nutrients": NUTRIENTS,
}

MODEL_PATH = "/content/kisanai_models/fertilizer_model.pkl"
joblib.dump(artifact, MODEL_PATH)
print("Saved:", MODEL_PATH)

## FertilizerAgent class -- ready for the Orchestrator

Matches the style of `CropAgent` / `WeatherAgent` from `05_KisanAI_Orchestrator.ipynb`
so it can be added to that orchestrator directly.

In [ ]:
class FertilizerAgent:
    def __init__(self, artifact):
        self.model = artifact["model"]
        self.encoder = artifact["encoder"]
        self.target_encoders = artifact["target_encoders"]
        self.context_numeric = artifact["context_numeric"]
        self.context_categorical = artifact["context_categorical"]
        self.value_cols = artifact["value_cols"]
        self.target_cols = artifact["target_cols"]
        self.nutrients = artifact["nutrients"]

    def recommend(self, data: dict) -> dict:
        """
        data must contain: pH, EC_dS_per_m, OC_percent, Soil_Type, Season,
        and the raw nutrient value columns (N_kg_per_ha, P_kg_per_ha, ... Mn_ppm).
        """
        row = pd.DataFrame([data])
        cat_part = pd.DataFrame(
            self.encoder.transform(row[self.context_categorical]),
            columns=self.encoder.get_feature_names_out(self.context_categorical)
        )
        x = pd.concat(
            [row[self.context_numeric].reset_index(drop=True),
             row[self.value_cols].reset_index(drop=True),
             cat_part],
            axis=1
        )
        pred_encoded = self.model.predict(x)[0]

        ratings = {}
        for i, col in enumerate(self.target_cols):
            nut = col.replace("_Rating", "")
            le = self.target_encoders[col]
            ratings[nut] = le.inverse_transform([pred_encoded[i]])[0]

        return {
            "ratings": ratings,
            "dosage_recommendation": dosage_recommendation(ratings),
        }


# Quick manual test using one real row from the test set
agent = FertilizerAgent(artifact)
sample_row = df.iloc[0]
sample_input = {
    **{c: sample_row[c] for c in CONTEXT_NUMERIC + CONTEXT_CATEGORICAL},
    **{c: sample_row[c] for c in VALUE_COLS},
}
result = agent.recommend(sample_input)
print("Ratings:", result["ratings"])
print("\nDosage recommendation:")
for nut, advice in result["dosage_recommendation"].items():
    print(f"  {nut}: {advice}")

## Download model for the local KisanAI project

In [ ]:
from google.colab import files
files.download(MODEL_PATH)

## Next integration step
Add `FertilizerAgent` to `05_KisanAI_Orchestrator.ipynb` alongside `CropAgent`, and route
an intent (e.g. `"fertilizer"`) to it in `KisanOrchestrator.route()`.